<a href="https://colab.research.google.com/github/getcommunityone/open-navigator/blob/main/scripts/colab/02_run_meeting_llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 · Run meeting LLM — 2026 Gemma 4 Good Hackathon

To make those demo cells work without throwing errors, follow this workflow:

- Get your Key from AI Studio
- Go to [Google AI Studio](https://aistudio.google.com).
- Click Get API Key and copy the string.
- Check your project dashboard if you need to find a specific model ID to replace the default Gemma 4 alias.
- Set up secret in collab for GEMINI_API_KEY


This notebook walks `01_raw_inputs/<STATE>/<county|municipality>/<jurisdiction>/…` and
demonstrates **four Gemma 4 capabilities** on the discovered PDFs and audio:

1. **Native multimodality / visual document parsing** — Gemma 4 reads scanned
   municipal minutes (zero digital text) directly as images. Demoed on every PDF
   under `01_raw_inputs`; we flag scans where `pdftotext` returns nothing.
2. **Adjustable visual token budget** — the script classifies each PDF page and
   sends financial tables / scanned pages at **HIGH** (~1,120 tokens/image),
   text-heavy minutes at **LOW** (~64 tokens). Hackathon talking points:
   Big Timber Sweet Grass County Courthouse Project rankings; Big Timber Airport
   paving and gate-upgrade contract amounts; Tuscaloosa city/county routine minutes.
3. **Built-in thinking (reasoning) mode** — runs `prompts/policy_analysis_v1.md` with
   `include_thoughts=True` so judges can see the prevailing narrative vs. dissenting
   diagnosis (e.g. April 2026 Tuscaloosa County nuisance demolitions: collective
   safety + blight removal vs. individual property rights).
4. **Alternating local sliding-window + global attention** — chunks long audio
   into 15-minute pieces, analyses each, then runs a **Policy Drift Detector**
   pass that tracks how a subject (e.g. Tuscaloosa downtown zoning) shifts from
   origination through final amendment.

Outputs mirror the input layout under `03_processed_outputs/02_gemma_json/<STATE>/<scope>/<jurisdiction>/…`
and `03_processed_outputs/03_human_summaries/…` so the geography encoded in the
folder names is preserved downstream.

## 1) Bootstrap repo + Drive

On Colab we mount Google Drive so the
pipeline data root (`CommunityOne/governance_pipeline_data/`) is visible; locally
we just point at the repo checkout. `OPEN_NAVIGATOR_ROOT` overrides discovery.

In [3]:
# Find the open-navigator repo so we can import shared helpers.
import os
import sys
from pathlib import Path

# --- Start of suggested fix ---
# Ensure the open-navigator repository is cloned and accessible.
repo_path = Path("/content/open-navigator")
if not repo_path.is_dir():
    print("Cloning open-navigator repository...")
    # Use os.system to run shell commands in Colab.
    exit_code = os.system(f"git clone https://github.com/getcommunityone/open-navigator.git {repo_path}")
    if exit_code != 0:
        raise RuntimeError(f"Failed to clone open-navigator repository. Exit code: {exit_code}")
    print(f"Repository cloned to {repo_path}")

# Set OPEN_NAVIGATOR_ROOT environment variable if it's not already set or is empty.
if not os.environ.get("OPEN_NAVIGATOR_ROOT") or os.environ.get("OPEN_NAVIGATOR_ROOT").strip() == "":
    os.environ["OPEN_NAVIGATOR_ROOT"] = str(repo_path)
# --- End of suggested fix ---


def _bootstrap_repo() -> Path:
    env = os.environ.get("OPEN_NAVIGATOR_ROOT", "").strip()
    if env:
        p = Path(env).expanduser().resolve()
        if (p / "scripts" / "colab" / "colab_paths.py").is_file():
            s = str(p)
            if s not in sys.path:
                sys.path.insert(0, s)
            return p
    for anchor in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (anchor / "scripts" / "colab" / "colab_paths.py").is_file():
            s = str(anchor)
            if s not in sys.path:
                sys.path.insert(0, s)
            return anchor
    raise RuntimeError(
        "Could not find open-navigator (scripts/colab/colab_paths.py). "
        "cd to the repo root before starting Jupyter, or set OPEN_NAVIGATOR_ROOT."
    )


_bootstrap_repo()

from scripts.colab.colab_paths import maybe_mount_google_drive, setup_notebook_paths

maybe_mount_google_drive()
PATHS = setup_notebook_paths()
print("Runtime:", "Google Colab" if PATHS.in_colab else "Local")
print("Repo:                  ", PATHS.project_path)
print("Governance data root: ", PATHS.governance_pipeline_data)


Cloning open-navigator repository...
Repository cloned to /content/open-navigator
Mounted at /content/drive
Runtime: Google Colab
Repo:                   /content/open-navigator
Governance data root:  /content/governance_pipeline_data


In [4]:
import os
import sys

# Flip to False if you do not want a hard reset to origin/main.
RUN_GIT_UPDATE = True

proj = PATHS.project_path
if RUN_GIT_UPDATE and proj.is_dir() and (proj / ".git").is_dir():
    print("🔄 Fetching latest code from GitHub...")
    !cd {proj} && git config core.hooksPath /dev/null && git fetch origin && git reset --hard origin/main
elif RUN_GIT_UPDATE:
    print("⚠️ Skip git: not a git checkout at", proj)

assert (proj / "prompts" / "policy_analysis_v1.md").is_file(), (
    f"Missing {proj}/prompts/policy_analysis_v1.md"
)

sys.path.insert(0, str(proj))
COLAB_DIR = proj / "scripts" / "colab"
sys.path.insert(0, str(COLAB_DIR))

os.environ["GOVERNANCE_PIPELINE_DATA_ROOT"] = str(PATHS.governance_pipeline_data)

from scripts.utils.gdrive_paths import GovernancePipelinePaths

PIPE = GovernancePipelinePaths.resolve()
PIPE.ensure_dirs()

from governance_meeting_llm import (
    AUDIO_EXTS,
    PDF_EXTS,
    TOKEN_BUDGET_HIGH,
    TOKEN_BUDGET_LOW,
    call_google_genai_multimodal,
    chunk_audio_ffmpeg,
    classify_pdf_page_heuristic,
    extract_pdf_digital_text,
    is_scanned_pdf,
    load_contacts_lookup,
    load_meeting_data_lookup,
    load_text_file,
    merge_contacts_by_jurisdiction,
    merge_meeting_data_by_jurisdiction,
    mime_for,
    mirror_output_path,
    parse_policy_analysis_response,
    policy_drift_summarize,
    render_pdf_pages,
    walk_raw_inputs,
)

PROMPT_PATH = proj / "prompts" / "policy_analysis_v1.md"
POLICY_PROMPT = load_text_file(PROMPT_PATH)
print("Loaded prompt chars:", len(POLICY_PROMPT))
print("Pipeline data root:", PIPE.root)

🔄 Fetching latest code from GitHub...
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 6.31 KiB | 1.05 MiB/s, done.
From https://github.com/getcommunityone/open-navigator
   1076cbf..9a41d37  main       -> origin/main
Updating files: 100% (21/21), done.
HEAD is now at 9a41d37 Created using Colab
Encountered 20 file(s) that should have been pointers, but weren't:
	frontend/public/wikicommons/GA_latest.jpg
	frontend/public/wikicommons/GU_latest.png
	frontend/public/wikicommons/HI_latest.jpg
	frontend/public/wikicommons/IA_latest.jpg
	frontend/public/wikicommons/ID_latest.jpg
	frontend/public/wikicommons/IL_latest.jpg
	frontend/public/wikicommons/IN_latest.jpg
	frontend/public/wikicommons/KS_latest.jpg
	frontend/public/wikicommons/KY_latest.jpg
	frontend/public/wikicommons/LA_latest.jpg
	website/static/img/comm

## 2) Install dependencies

`google-genai` is the Gemini / Gemma 4 client. `pymupdf` renders PDF pages to
images and probes their digital-text layer for the per-page token-budget demo.
`pdf2image` (used by the Gatekeeper step 0) needs the system package
`poppler-utils` — pre-installed on Colab; locally `apt-get install -y poppler-utils`
on Debian/Ubuntu or `brew install poppler` on macOS. `ffmpeg` ships with Colab;
locally `apt-get install -y ffmpeg` if you plan to run **Demo 4** (audio chunking)
or the Gatekeeper audio gate.

In [5]:
# poppler-utils is required by pdf2image (Gatekeeper step 0). Skipped silently off Colab.
import shutil, subprocess
if not shutil.which("pdftoppm"):
    try:
        subprocess.run(["apt-get", "-qq", "install", "-y", "poppler-utils"], check=False)
    except FileNotFoundError:
        pass

%pip install -q -U "google-genai>=1.0" pymupdf pdf2image

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 728.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 805.5/805.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.1/246.1 kB 13.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.53.0 which is incompatible.
google-cloud-aiplatform 1.148.1 requires google-genai<2.0.0,>=1.66.0; python_version >= "3.10", but you have google-genai 2.3.0 which is incompatible.
google-adk 1.29.0 requires google-genai<2.0.0,>=1.64.0, but you have google-genai 2.3.0 which is incompatible.


## 3) API key + model + demo caps

API key resolution: Colab **Secret** named `GEMINI_API_KEY` (preferred) → env
`GEMINI_API_KEY` / `GOOGLE_API_KEY`. Model id defaults to a Gemma 4 alias; override
with `GOVERNANCE_GENAI_MODEL` if your AI Studio project lists a different id.

The four demo cells honor a few env knobs so live runs don't blow through tokens:

| Env var | Default | What it caps |
|---|---|---|
| `GOVERNANCE_DEMO_MAX_PDFS_PER_JUR` | `3` | PDFs processed per jurisdiction in Demo 1 + 2 |
| `GOVERNANCE_DEMO_MAX_PAGES_PER_PDF` | `8` | Pages processed per PDF in Demo 2 |
| `GOVERNANCE_DEMO_MAX_AUDIO_PER_JUR` | `1` | Audio files processed per jurisdiction in Demo 4 |
| `GOVERNANCE_DEMO_MAX_AUDIO_CHUNKS` | `4` | 15-minute audio chunks processed per file in Demo 4 |
| `GOVERNANCE_DEMO_THINKING_BUDGET` | `-1` | Thinking-token budget for Demo 3 (`-1` = unlimited) |

In [6]:
import os

try:
    from google.colab import userdata

    _get_secret = userdata.get
except ImportError:

    def _get_secret(name: str):
        return os.environ.get(name)


GEMINI_API_KEY = (
    _get_secret("GEMINI_API_KEY")
    or _get_secret("GOOGLE_API_KEY")
    or os.environ.get("GEMINI_API_KEY")
    or os.environ.get("GOOGLE_API_KEY")
)
if not GEMINI_API_KEY:
    raise RuntimeError(
        "Set GEMINI_API_KEY (Colab Secret or environment). "
        "Create a key at https://aistudio.google.com/apikey"
    )

# Default is a Gemma 4 alias — override via env if your AI Studio project lists a different id.
GENAI_MODEL = os.environ.get("GOVERNANCE_GENAI_MODEL", "gemma-4-26b-a4b-it").strip()

# Demo caps (override via env to widen / narrow scope).
MAX_PDFS_PER_JUR = int(os.environ.get("GOVERNANCE_DEMO_MAX_PDFS_PER_JUR", "3"))
MAX_PAGES_PER_PDF = int(os.environ.get("GOVERNANCE_DEMO_MAX_PAGES_PER_PDF", "8"))
MAX_AUDIO_PER_JUR = int(os.environ.get("GOVERNANCE_DEMO_MAX_AUDIO_PER_JUR", "1"))
MAX_AUDIO_CHUNKS = int(os.environ.get("GOVERNANCE_DEMO_MAX_AUDIO_CHUNKS", "4"))
THINKING_BUDGET = int(os.environ.get("GOVERNANCE_DEMO_THINKING_BUDGET", "-1"))
DRIFT_FOCUS = os.environ.get("GOVERNANCE_DRIFT_FOCUS", "").strip() or None

print("GenAI model:", GENAI_MODEL)
print(
    "Caps:",
    f"pdfs/jur={MAX_PDFS_PER_JUR}",
    f"pages/pdf={MAX_PAGES_PER_PDF}",
    f"audio/jur={MAX_AUDIO_PER_JUR}",
    f"chunks/audio={MAX_AUDIO_CHUNKS}",
    f"thinking_budget={THINKING_BUDGET}",
)

GenAI model: gemma-4-26b-a4b-it
Caps: pdfs/jur=3 pages/pdf=8 audio/jur=1 chunks/audio=4 thinking_budget=-1


## 4) Step 0 — Gatekeeper Triage (data gate)

> **The Ledger of Influence — data gate.** Before any expensive deconstruction
> pass, route every raw input through `gatekeeper_triage.run_triage()`. The
> Gatekeeper sends each PDF and audio file to Gemma for a cheap, strict-JSON
> verdict (`is_governance_meeting`, `document_or_audio_type`, `confidence_score`,
> `reasoning`). Files the model rejects are moved to
> `01_raw_inputs/excluded_inputs/<STATE>/<scope>/<jurisdiction>/…` — the original
> geography subtree is replicated so we can audit *why* every file was kept or
> dropped.

Knobs:

| Env var | Default | Effect |
|---|---|---|
| `GOVERNANCE_GATEKEEPER_ENABLED` | `1` | Set `0` to skip the gate entirely. |
| `GOVERNANCE_GATEKEEPER_DRY_RUN` | `0` | Set `1` for a no-move audit pass — verdicts logged, nothing relocated. |
| `GOVERNANCE_GATEKEEPER_KINDS` | `pdf,audio` | Limit to one kind during demo runs. |
| `GOVERNANCE_GATEKEEPER_AUDIO_WINDOW` | `120` | Seconds of audio sent to triage (programmatic stream-depth cap). |
| `GOVERNANCE_GATEKEEPER_PDF_PAGES` | `2` | First N PDF pages sent to triage at HIGH (~1,120 image tokens). |
| `GOVERNANCE_GATEKEEPER_CONFIDENCE` | `0.6` | Minimum confidence to keep a file. |
| `GOVERNANCE_GATEKEEPER_MAX_FILES` | (unset) | Stop after N files — useful for live demos. |

Re-runs are idempotent: the walker prunes `excluded_inputs/` from its descent so
already-rejected files are never re-triaged.

In [7]:
# Step 0 — Gatekeeper Triage. Routes raw inputs through Gemma before any deep pass.
import os
from pathlib import Path

import gatekeeper_triage

RAW_ROOT_FOR_GATE = Path(
    os.environ.get("GOVERNANCE_RAW_INPUTS_ROOT", "").strip() or (PIPE.root / "01_raw_inputs")
).expanduser()

GATEKEEPER_ENABLED = os.environ.get("GOVERNANCE_GATEKEEPER_ENABLED", "1") != "0"
GATEKEEPER_DRY_RUN = os.environ.get("GOVERNANCE_GATEKEEPER_DRY_RUN", "0") == "1"

if not GATEKEEPER_ENABLED:
    print("Gatekeeper disabled (GOVERNANCE_GATEKEEPER_ENABLED=0). Skipping triage.")
elif not RAW_ROOT_FOR_GATE.is_dir():
    print(f"Skipping Gatekeeper: raw inputs root missing — {RAW_ROOT_FOR_GATE}")
else:
    kinds = tuple(
        k.strip().lower()
        for k in os.environ.get("GOVERNANCE_GATEKEEPER_KINDS", "pdf,audio").split(",")
        if k.strip()
    )
    max_files_env = os.environ.get("GOVERNANCE_GATEKEEPER_MAX_FILES", "").strip()
    max_files = int(max_files_env) if max_files_env else None

    print(f"Gatekeeper sweep | raw_root={RAW_ROOT_FOR_GATE} | kinds={kinds} | dry_run={GATEKEEPER_DRY_RUN}")
    gatekeeper_triage._configure_logging(verbose=False)
    report = gatekeeper_triage.run_triage(
        raw_root=RAW_ROOT_FOR_GATE,
        api_key=GEMINI_API_KEY,
        model=GENAI_MODEL,
        kinds=kinds,
        pdf_pages=int(os.environ.get("GOVERNANCE_GATEKEEPER_PDF_PAGES", "2")),
        audio_window_seconds=int(os.environ.get("GOVERNANCE_GATEKEEPER_AUDIO_WINDOW", "120")),
        confidence_threshold=float(os.environ.get("GOVERNANCE_GATEKEEPER_CONFIDENCE", "0.6")),
        dry_run=GATEKEEPER_DRY_RUN,
        max_files=max_files,
    )

    # Persist the report alongside the other processed outputs for audit / demo.
    report_dir = PIPE.root / "03_processed_outputs" / "_gatekeeper"
    report_dir.mkdir(parents=True, exist_ok=True)
    from datetime import datetime, timezone
    import json as _json

    stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    report_path = report_dir / f"triage_report_{stamp}.json"
    report_path.write_text(_json.dumps(report.to_dict(), indent=2, ensure_ascii=False), encoding="utf-8")

    print(
        f"\nGatekeeper done | proceed={len(report.proceed)} | excluded={len(report.excluded)} "
        f"| errors={len(report.errors)} | report → {report_path.relative_to(PIPE.root)}"
    )
    if report.excluded:
        print("\nExcluded files (mirrored under excluded_inputs/<state>/<scope>/<jurisdiction>/…):")
        for v in report.excluded[:10]:
            print(f"  · {v.relative_path}: {v.document_or_audio_type} ({v.confidence_score:.2f}) — {v.reasoning[:140]}")
        if len(report.excluded) > 10:
            print(f"  …and {len(report.excluded) - 10} more (see report).")

Gatekeeper sweep | raw_root=/content/governance_pipeline_data/01_raw_inputs | kinds=('pdf', 'audio') | dry_run=False

Gatekeeper done | proceed=0 | excluded=0 | errors=0 | report → 03_processed_outputs/_gatekeeper/triage_report_20260516_113950.json


## 5) Walk `01_raw_inputs` by state / scope / jurisdiction

The walker honors the layout the sync script writes:
`01_raw_inputs/<STATE>/<county|municipality>/<jurisdiction_slug>/…` with FIPS
encoded in the slug (`county_01125` = Tuscaloosa County AL FIPS 01125;
`municipality_3006475` = Big Timber MT). Scraper-internal folders prefixed with
underscore (`_crawl_html`, `_sitemaps`, `_contact_images`) are skipped.

The cell below prints an inventory and stores it in `INVENTORIES` for the four
demo cells. Set `GOVERNANCE_RAW_INPUTS_ROOT` to point at a different root if your
raw inputs live outside the standard pipeline path.

In [8]:
from pathlib import Path
import os

RAW_ROOT = Path(
    os.environ.get("GOVERNANCE_RAW_INPUTS_ROOT", "").strip() or (PIPE.root / "01_raw_inputs")
).expanduser()
PROCESSED_ROOT = PIPE.root / "03_processed_outputs"
GEMMA_JSON_ROOT = PROCESSED_ROOT / "02_gemma_json"
SUMMARIES_ROOT = PROCESSED_ROOT / "03_human_summaries"
SCRATCH_AUDIO_ROOT = PROCESSED_ROOT / "_scratch_audio_chunks"

for p in (GEMMA_JSON_ROOT, SUMMARIES_ROOT, SCRATCH_AUDIO_ROOT):
    p.mkdir(parents=True, exist_ok=True)

if not RAW_ROOT.is_dir():
    raise FileNotFoundError(
        f"Raw inputs root missing: {RAW_ROOT}. Run 01_copy_scraped_meetings_cache_to_gdrive.py "
        "or set GOVERNANCE_RAW_INPUTS_ROOT."
    )

INVENTORIES = [inv for inv in walk_raw_inputs(RAW_ROOT) if inv.has_media]

print(f"Raw inputs root: {RAW_ROOT}")
print(f"Jurisdictions with media: {len(INVENTORIES)}")
for inv in INVENTORIES:
    j = inv.jurisdiction
    fips = j.fips or "—"
    print(
        f"  {j.relative_label}  fips={fips}  "
        f"pdfs={len(inv.pdfs)}  audio={len(inv.audio)}  images={len(inv.images)}"
    )

if not INVENTORIES:
    print(
        "\nNo media found. Drop PDFs / audio under "
        f"{RAW_ROOT}/<STATE>/<county|municipality>/<jurisdiction_slug>/…"
    )

Raw inputs root: /content/governance_pipeline_data/01_raw_inputs
Jurisdictions with media: 0

No media found. Drop PDFs / audio under /content/governance_pipeline_data/01_raw_inputs/<STATE>/<county|municipality>/<jurisdiction_slug>/…


## 6) Demo 1 — Native multimodality / visual document parsing

> **Dark-data problem:** Many municipal minutes from Tuscaloosa and Big Timber are
> physically printed, signed, and run through an office scanner — the resulting
> PDF contains **zero** digital text. Traditional text extractors return empty
> strings. Gemma 4 reads the page as an image and OCRs it natively.

For every PDF discovered by the walker, this cell:

1. Probes the digital-text layer with PyMuPDF (`pdftotext`-equivalent).
2. Tags PDFs with `<200` chars of extractable text as **scanned (dark-data)**.
3. Sends the PDF bytes to Gemma 4 with a minimal "extract everything" prompt.
4. Writes the model's transcribed text to
   `03_processed_outputs/02_gemma_json/<STATE>/<scope>/<jurisdiction>/<path>.visual_ocr.txt`.

The summary table front-loads the scanned-vs-digital split so judges immediately
see the dark-data win.

In [ ]:
# Demo 1 — Native multimodality / visual document parsing.
# Probes the digital-text layer of every PDF and uses Gemma 4 to OCR scans.
from datetime import datetime, timezone

DEMO1_SYSTEM = (
    "You are a careful document transcription engine. Faithfully transcribe every "
    "word and number on each page of the attached PDF in reading order. Preserve "
    "table structure with vertical bars and dashes. Do not paraphrase, summarize, "
    "or invent content."
)
DEMO1_USER = (
    "Transcribe the attached PDF page by page. Begin each page with a heading line "
    "'### Page <n>' on its own line. If a page is blank, write '(blank page)'."
)

demo1_report = []
for inv in INVENTORIES:
    j = inv.jurisdiction
    pdfs = inv.pdfs[:MAX_PDFS_PER_JUR]
    if not pdfs:
        continue
    print(f"\n— {j.relative_label} — {len(pdfs)} PDF(s)")
    for pdf in pdfs:
        try:
            digital_chars = len(extract_pdf_digital_text(pdf))
        except Exception as e:
            print(f"  ! {pdf.name}: PDF probe failed — {e}")
            continue
        scanned = digital_chars < 200
        tag = "SCANNED (dark data)" if scanned else f"digital ({digital_chars} chars)"
        print(f"  • {pdf.name}: {tag}")

        try:
            result = call_google_genai_multimodal(
                api_key=GEMINI_API_KEY,
                model=GENAI_MODEL,
                system_instruction=DEMO1_SYSTEM,
                user_text=DEMO1_USER,
                media=[(pdf, "application/pdf")],
                temperature=0.0,
                max_output_tokens=8192,
                media_resolution=TOKEN_BUDGET_HIGH if scanned else None,
            )
        except Exception as e:
            print(f"    ! Gemma call failed: {e}")
            continue

        out_txt = mirror_output_path(
            input_path=pdf,
            raw_root=RAW_ROOT,
            processed_root=GEMMA_JSON_ROOT,
            suffix=".visual_ocr.txt",
        )
        out_txt.write_text(result.text or "(empty response)", encoding="utf-8")
        demo1_report.append(
            {
                "jurisdiction": j.relative_label,
                "fips": j.fips,
                "pdf": str(pdf.relative_to(RAW_ROOT)),
                "scanned": scanned,
                "digital_chars": digital_chars,
                "output": str(out_txt.relative_to(PROCESSED_ROOT)),
                "model_chars": len(result.text or ""),
            }
        )
        print(f"    → {out_txt.relative_to(PROCESSED_ROOT)} ({len(result.text or '')} chars)")

scanned_count = sum(1 for r in demo1_report if r["scanned"])
print(
    f"\nDemo 1 done: {len(demo1_report)} PDFs, {scanned_count} flagged scanned "
    "→ Gemma 4 visual OCR recovered text that pdftotext could not."
)

import json as _json
demo1_summary_path = GEMMA_JSON_ROOT / "_demo1_visual_ocr_report.json"
demo1_summary_path.write_text(_json.dumps(demo1_report, indent=2), encoding="utf-8")
print("Report:", demo1_summary_path.relative_to(PROCESSED_ROOT))

## 7) Demo 2 — Adjustable visual token budget per page

> **Why:** Gemma 4 lets us spend up to ~1,120 image tokens per page when we need
> ultra-fine detail (financial ledgers, contract bid alignments, scanned forms),
> and drop to ~64 tokens per page on text-heavy minutes to protect runtime and
> API cost.

For each PDF the cell renders every page to a PNG, classifies it via
`classify_pdf_page_heuristic`, and routes:

- **`scanned`** → **HIGH** budget — needs visual OCR to recover any text at all.
- **`financial_or_tabular`** → **HIGH** budget — Big Timber courthouse architect
  rankings or airport paving contract awards demand column alignment fidelity.
- **`text_heavy`** → **LOW** budget — standard council minutes; speed + cost win.

Each page produces one JSON file under
`02_gemma_json/<mirrored>/<pdf_stem>/page_<n>.json` plus a per-PDF
`_token_budget_report.json` summarizing the budget split.

In [ ]:
# Demo 2 — Adjustable visual token budget per page.
# Classify each PDF page and route HIGH (~1,120 tokens) vs LOW (~64 tokens).
import json as _json
import time

DEMO2_SYSTEM = (
    "You are a careful page-level extractor. Return JSON only — no markdown fences."
)
DEMO2_USER_HIGH = (
    "This page contains tabular or financial content (bids, contract awards, "
    "ledgers, line-item budgets). Preserve column alignment and every dollar "
    "amount. Return JSON with this shape: "
    '{"page_type":"financial_or_tabular","raw_text":"...","line_items":[{"label":"...","amount":"..."}],"notes":"..."}'
)
DEMO2_USER_LOW = (
    "This page is standard meeting minutes text. Return JSON with this shape: "
    '{"page_type":"text_heavy","raw_text":"...","headline":"...","notes":"..."}'
)
DEMO2_USER_SCANNED = (
    "This page is a scanned image with no digital text. Visually OCR it. "
    "Return JSON with this shape: "
    '{"page_type":"scanned","raw_text":"...","notes":"..."}'
)
USER_BY_CLASS = {
    "financial_or_tabular": DEMO2_USER_HIGH,
    "text_heavy": DEMO2_USER_LOW,
    "scanned": DEMO2_USER_SCANNED,
}

demo2_report = []
for inv in INVENTORIES:
    j = inv.jurisdiction
    pdfs = inv.pdfs[:MAX_PDFS_PER_JUR]
    for pdf in pdfs:
        print(f"\n— {j.relative_label} / {pdf.name}")
        try:
            pages = render_pdf_pages(pdf, dpi=200)
        except Exception as e:
            print(f"  ! render failed: {e}")
            continue
        pages = pages[:MAX_PAGES_PER_PDF]

        per_pdf_dir = mirror_output_path(
            input_path=pdf,
            raw_root=RAW_ROOT,
            processed_root=GEMMA_JSON_ROOT,
            suffix="",
        )
        per_pdf_dir.mkdir(parents=True, exist_ok=True)
        if per_pdf_dir.is_file():
            per_pdf_dir.unlink()
            per_pdf_dir.mkdir(parents=True, exist_ok=True)

        pdf_summary = {
            "jurisdiction": j.relative_label,
            "fips": j.fips,
            "pdf": str(pdf.relative_to(RAW_ROOT)),
            "page_count": len(pages),
            "budget_split": {TOKEN_BUDGET_HIGH: 0, TOKEN_BUDGET_LOW: 0},
            "pages": [],
        }
        for page in pages:
            budget = page.token_budget
            user = USER_BY_CLASS.get(page.classification, DEMO2_USER_LOW)
            t0 = time.time()
            try:
                result = call_google_genai_multimodal(
                    api_key=GEMINI_API_KEY,
                    model=GENAI_MODEL,
                    system_instruction=DEMO2_SYSTEM,
                    user_text=user,
                    media=[(page.image_bytes, "image/png")],
                    temperature=0.0,
                    max_output_tokens=2048,
                    media_resolution=budget,
                )
            except Exception as e:
                print(f"  ! page {page.page_index + 1}: Gemma call failed — {e}")
                continue
            elapsed = time.time() - t0

            try:
                page_json = _json.loads(result.text.strip().lstrip("`"))
            except Exception:
                page_json = {"_parse_error": True, "_raw": result.text[:2000]}

            page_out = per_pdf_dir / f"page_{page.page_index + 1:03d}.json"
            page_out.write_text(
                _json.dumps(
                    {
                        "page_index": page.page_index,
                        "classification": page.classification,
                        "token_budget": budget,
                        "elapsed_s": round(elapsed, 2),
                        "model": GENAI_MODEL,
                        "extracted": page_json,
                    },
                    indent=2,
                    ensure_ascii=False,
                ),
                encoding="utf-8",
            )
            pdf_summary["budget_split"][budget] = pdf_summary["budget_split"].get(budget, 0) + 1
            pdf_summary["pages"].append(
                {
                    "page": page.page_index + 1,
                    "classification": page.classification,
                    "token_budget": budget,
                    "elapsed_s": round(elapsed, 2),
                }
            )
            print(
                f"  page {page.page_index + 1}: {page.classification:>22}  "
                f"→ {budget:<6} ({elapsed:.1f}s)"
            )

        report_path = per_pdf_dir / "_token_budget_report.json"
        report_path.write_text(_json.dumps(pdf_summary, indent=2), encoding="utf-8")
        demo2_report.append(pdf_summary)
        high = pdf_summary["budget_split"].get(TOKEN_BUDGET_HIGH, 0)
        low = pdf_summary["budget_split"].get(TOKEN_BUDGET_LOW, 0)
        print(f"  budget split: HIGH={high}  LOW={low}  → report {report_path.relative_to(PROCESSED_ROOT)}")

if demo2_report:
    total_high = sum(r["budget_split"].get(TOKEN_BUDGET_HIGH, 0) for r in demo2_report)
    total_low = sum(r["budget_split"].get(TOKEN_BUDGET_LOW, 0) for r in demo2_report)
    total = total_high + total_low or 1
    print(
        f"\nDemo 2 done: {total} pages across {len(demo2_report)} PDF(s). "
        f"HIGH-budget pages: {total_high} ({100 * total_high // total}%). "
        f"LOW-budget pages: {total_low} ({100 * total_low // total}%). "
        "Dropping text-heavy minutes to LOW is the cost / latency win we show judges."
    )
else:
    print("\nDemo 2: no PDFs processed (check MAX_PDFS_PER_JUR and inventory above).")

## 8) Demo 3 — Built-in thinking on the deconstruction prompt

> **Hackathon talking point:** April 2026 Tuscaloosa County nuisance property
> demolitions. Gemma 4's multi-step planning surfaces the prevailing narrative
> (collective safety + blight removal) and the dissenting diagnosis (individual
> property rights from protesting neighbors) in the same pass — not just a
> chronological summary.

For each jurisdiction the cell picks a representative PDF (largest by page count,
preferring filenames that suggest minutes / hearings) and runs the full
`prompts/policy_analysis_v1.md` prompt with `include_thoughts=True` and an
extended `thinking_budget`. Outputs:

- `…/02_gemma_json/<mirrored>/<pdf_stem>.thinking.json` — parsed JSON analysis.
- `…/02_gemma_json/<mirrored>/<pdf_stem>.thinking.raw.txt` — raw model output.
- `…/02_gemma_json/<mirrored>/<pdf_stem>.thinking.thoughts.md` — reasoning trace.
- `…/03_human_summaries/<mirrored>/<pdf_stem>.thinking.summary.md` — human summary.

In [ ]:
# Demo 3 — Built-in thinking mode on the deconstruction prompt.
# One representative PDF per jurisdiction is processed with include_thoughts=True.
import json as _json

DEMO3_SYSTEM = (
    "You are an expert political scientist and data architect specializing in "
    "local governance. Follow the user's instructions exactly; preserve JSON validity."
)

_PRIORITY_PATTERNS = (
    "demolition", "demolitions", "nuisance",
    "minutes", "regular_session", "regular-session", "regular session",
    "council", "commission", "board", "hearing",
)


def _pick_representative_pdf(pdfs):
    if not pdfs:
        return None
    scored = []
    for p in pdfs:
        name = p.name.lower()
        score = 0
        for tag in _PRIORITY_PATTERNS:
            if tag in name:
                score += 5
        try:
            score += min(p.stat().st_size // 50_000, 50)
        except OSError:
            pass
        scored.append((score, p))
    scored.sort(key=lambda t: (-t[0], t[1].name))
    return scored[0][1]


demo3_report = []
for inv in INVENTORIES:
    j = inv.jurisdiction
    pdf = _pick_representative_pdf(inv.pdfs)
    if pdf is None:
        continue
    print(f"\n— {j.relative_label}: {pdf.name}")

    geo_hint = (
        f"Geography hint from folder layout: state_code={j.state_code}, "
        f"scope={j.scope}, fips_or_place_id={j.fips or 'unknown'}. "
        "Use this when populating county_fips / county / postal_code in each decision."
    )
    user_text = (
        f"{POLICY_PROMPT}\n\n---\n{geo_hint}\n\n"
        "The attached PDF contains the meeting record. Apply the full deconstruction "
        "prompt to it. Stick to what is actually in the document."
    )

    try:
        result = call_google_genai_multimodal(
            api_key=GEMINI_API_KEY,
            model=GENAI_MODEL,
            system_instruction=DEMO3_SYSTEM,
            user_text=user_text,
            media=[(pdf, "application/pdf")],
            temperature=0.1,
            max_output_tokens=8192,
            media_resolution=TOKEN_BUDGET_HIGH,
            include_thoughts=True,
            thinking_budget=THINKING_BUDGET,
        )
    except Exception as e:
        print(f"  ! Gemma call failed: {e}")
        continue

    parsed = parse_policy_analysis_response(result.text or "")

    raw_out = mirror_output_path(
        input_path=pdf, raw_root=RAW_ROOT, processed_root=GEMMA_JSON_ROOT,
        suffix=".thinking.raw.txt",
    )
    raw_out.write_text(result.text or "", encoding="utf-8")

    if parsed.get("json_analysis") is not None:
        json_out = mirror_output_path(
            input_path=pdf, raw_root=RAW_ROOT, processed_root=GEMMA_JSON_ROOT,
            suffix=".thinking.json",
        )
        json_out.write_text(
            _json.dumps(parsed["json_analysis"], indent=2, ensure_ascii=False),
            encoding="utf-8",
        )
        print(f"  → {json_out.relative_to(PROCESSED_ROOT)}")

    if parsed.get("summary"):
        summary_out = mirror_output_path(
            input_path=pdf, raw_root=RAW_ROOT, processed_root=SUMMARIES_ROOT,
            suffix=".thinking.summary.md",
        )
        summary_out.write_text(parsed["summary"], encoding="utf-8")
        print(f"  → {summary_out.relative_to(PROCESSED_ROOT)}")

    if result.thoughts:
        thoughts_out = mirror_output_path(
            input_path=pdf, raw_root=RAW_ROOT, processed_root=GEMMA_JSON_ROOT,
            suffix=".thinking.thoughts.md",
        )
        thoughts_out.write_text(result.thoughts, encoding="utf-8")
        print(
            f"  → {thoughts_out.relative_to(PROCESSED_ROOT)} "
            f"(reasoning trace: {len(result.thoughts)} chars)"
        )
    else:
        print("  (no thoughts returned — model may not have surfaced a trace this run)")

    demo3_report.append(
        {
            "jurisdiction": j.relative_label,
            "pdf": str(pdf.relative_to(RAW_ROOT)),
            "thoughts_chars": len(result.thoughts or ""),
            "json_ok": parsed.get("json_analysis") is not None and "_error" not in (parsed.get("json_analysis") or {}),
            "parse_error": parsed.get("parse_error"),
        }
    )

if demo3_report:
    ok = sum(1 for r in demo3_report if r["json_ok"])
    print(
        f"\nDemo 3 done: {len(demo3_report)} PDFs deconstructed, "
        f"{ok} produced parseable JSON. Reasoning trace saved alongside each output."
    )
else:
    print("\nDemo 3: no PDFs available — check the walker output above.")

## 9) Demo 4 — Long-meeting chunking + Policy Drift Detector

> **Why:** Governance meeting videos often run 2–4 hours. We feed the audio in
> **15-minute chunks** to Gemma 4 and rely on the model's alternating local
> sliding-window + global attention to keep the broader thread across the meeting.
> The drift detector then runs a second Gemma pass that takes every chunk's JSON
> output and reports how a specific subject (e.g. Tuscaloosa downtown zoning)
> migrates from origination to final amendment.

For each audio file the cell:

1. Splits the file with `ffmpeg` into 15-minute MP3 chunks under a scratch dir.
2. Runs `prompts/policy_analysis_v1.md` per chunk, attaching the audio chunk.
3. Stores per-chunk JSON under `02_gemma_json/<mirrored>/<audio_stem>/chunk_<n>.json`.
4. Calls `policy_drift_summarize` with the assembled chunk JSONs and writes
   `…/<audio_stem>/policy_drift.json` (+ `.mmd` Mermaid timeline).

Set `GOVERNANCE_DRIFT_FOCUS` to bias the detector toward a specific subject string.

In [ ]:
# Demo 4 — Long-meeting chunking + Policy Drift Detector.
# Splits audio into 15-minute chunks, runs policy_analysis_v1 per chunk, then runs the drift pass.
import json as _json

DEMO4_SYSTEM = (
    "You are an expert political scientist analyzing one chunk of a long meeting. "
    "Follow the user's instructions exactly; preserve JSON validity. The chunk_index "
    "tells you which 15-minute slice of the meeting this audio covers."
)

demo4_report = []
for inv in INVENTORIES:
    j = inv.jurisdiction
    audios = inv.audio[:MAX_AUDIO_PER_JUR]
    if not audios:
        continue
    print(f"\n— {j.relative_label}: {len(audios)} audio file(s)")
    for audio in audios:
        print(f"  • {audio.name}")
        per_audio_dir = mirror_output_path(
            input_path=audio, raw_root=RAW_ROOT, processed_root=GEMMA_JSON_ROOT, suffix="",
        )
        per_audio_dir.mkdir(parents=True, exist_ok=True)

        # Scratch dir for ffmpeg chunks. Mirror layout so multiple meetings don't collide.
        rel = audio.resolve().relative_to(RAW_ROOT.resolve())
        scratch = SCRATCH_AUDIO_ROOT / rel.with_suffix("")
        scratch.mkdir(parents=True, exist_ok=True)

        try:
            chunks = chunk_audio_ffmpeg(audio, out_dir=scratch, chunk_minutes=15, fmt="mp3")
        except Exception as e:
            print(f"    ! ffmpeg chunking failed: {e}")
            continue
        chunks = chunks[:MAX_AUDIO_CHUNKS]
        print(f"    {len(chunks)} chunk(s) of 15 min each (cap MAX_AUDIO_CHUNKS={MAX_AUDIO_CHUNKS})")

        chunk_jsons = []
        for idx, chunk_path in enumerate(chunks):
            geo_hint = (
                f"Geography hint: state_code={j.state_code}, scope={j.scope}, "
                f"fips_or_place_id={j.fips or 'unknown'}. chunk_index={idx} of {len(chunks)}."
            )
            user_text = (
                f"{POLICY_PROMPT}\n\n---\n{geo_hint}\n\n"
                "The attached audio is one 15-minute slice of a longer governance meeting. "
                "Apply the deconstruction prompt to what is audible. Use the chunk_index "
                "to anchor the timeline and assign consistent subject_id slugs across chunks."
            )
            try:
                result = call_google_genai_multimodal(
                    api_key=GEMINI_API_KEY,
                    model=GENAI_MODEL,
                    system_instruction=DEMO4_SYSTEM,
                    user_text=user_text,
                    media=[(chunk_path, mime_for(chunk_path))],
                    temperature=0.1,
                    max_output_tokens=8192,
                )
            except Exception as e:
                print(f"    ! chunk {idx} failed: {e}")
                continue
            parsed = parse_policy_analysis_response(result.text or "")
            chunk_out = per_audio_dir / f"chunk_{idx:03d}.json"
            chunk_out.write_text(
                _json.dumps(
                    {
                        "chunk_index": idx,
                        "audio_source": str(chunk_path.name),
                        "json_analysis": parsed.get("json_analysis"),
                        "summary": parsed.get("summary"),
                        "parse_error": parsed.get("parse_error"),
                    },
                    indent=2,
                    ensure_ascii=False,
                ),
                encoding="utf-8",
            )
            chunk_jsons.append(parsed.get("json_analysis") or {})
            print(f"    chunk {idx}: → {chunk_out.relative_to(PROCESSED_ROOT)}")

        if not chunk_jsons:
            continue

        # Drift detector: Gemma 4 takes every chunk's JSON and reports per-subject
        # narrative drift along the five `narrative_analysis` dimensions defined in
        # `prompts/policy_analysis_v1.md` (dominant_narrative, dissenting_interpretations,
        # causal_interpretations, value_conflicts, tradeoff_analysis). The canonical
        # prompt body is pinned into context so the model honors the latest schema verbatim.
        drift = policy_drift_summarize(
            chunk_jsons,
            api_key=GEMINI_API_KEY,
            model=GENAI_MODEL,
            focus_hint=DRIFT_FOCUS,
            canonical_prompt_text=POLICY_PROMPT,
        )
        drift_out = per_audio_dir / "policy_drift.json"
        drift_out.write_text(_json.dumps(drift, indent=2, ensure_ascii=False), encoding="utf-8")

        drifted = drift.get("subjects") or drift.get("drifted_subjects") or []

        # Concatenate per-subject Mermaid timelines so the .mmd file is a single
        # diagram per audio (legacy schema returned one top-level diagram_timeline).
        mmd_blocks = []
        for s in drifted:
            tl = s.get("diagram_timeline")
            if isinstance(tl, str) and tl.strip():
                label = s.get("subject_label") or s.get("subject_id") or "subject"
                mmd_blocks.append(f"%% {label}\n{tl}")
        legacy_tl = drift.get("diagram_timeline")
        if not mmd_blocks and isinstance(legacy_tl, str) and legacy_tl.strip():
            mmd_blocks.append(legacy_tl)
        if mmd_blocks:
            (per_audio_dir / "policy_drift.mmd").write_text(
                "\n\n".join(mmd_blocks), encoding="utf-8"
            )

        meeting_summary = drift.get("meeting_level_summary") or {}
        print(
            f"    drift detector: {len(drifted)} subject(s) tracked across {len(chunk_jsons)} chunks "
            f"→ {drift_out.relative_to(PROCESSED_ROOT)}"
        )
        if meeting_summary.get("headline"):
            print(f"      meeting headline: {meeting_summary['headline'][:140]}")
        for s in drifted[:5]:
            label = s.get("subject_label") or s.get("subject_id") or "?"
            headline = s.get("drift_headline") or s.get("drift_summary") or ""
            stability = (s.get("narrative_stability_assessment") or {}).get("narrative_novelty")
            stability_tag = f" [{stability}]" if stability else ""
            print(f"      · {label}{stability_tag}: {headline[:120]}")
        demo4_report.append(
            {
                "jurisdiction": j.relative_label,
                "audio": str(audio.relative_to(RAW_ROOT)),
                "chunks": len(chunk_jsons),
                "subjects_tracked": len(drifted),
                "subjects_with_drift": sum(1 for s in drifted if s.get("drift_observed")),
                "emergent_value_tensions": meeting_summary.get("emergent_value_tensions") or [],
            }
        )

if demo4_report:
    print(
        f"\nDemo 4 done: {len(demo4_report)} audio file(s) processed. "
        "Gemma's alternating local + global attention kept subject ids consistent "
        "across 15-minute chunks; drift detector reported per-subject drift along the "
        "five narrative_analysis dimensions from policy_analysis_v1.md."
    )
else:
    print("\nDemo 4: no audio files in the inventory. Drop .mp3/.mp4/.wav under a jurisdiction folder.")

## 10) Demo 5 — Contact image enrichment (people vs. subjects)

> Each jurisdiction folder may carry a ``_contact_images/`` directory of scraped
> contact photos (council members, department heads, public officials). The walker
> exposes those images via ``inv.images``. This cell sends each image to Gemma 4
> vision with a strict-JSON prompt:
>
> - **If the image is a person** (public official in a public-records context):
>   return perceived ``age_range``, ``race``, ``gender``, ``ethnicity``, and
>   ``demeanor`` (e.g. ``formal``, ``smiling``, ``neutral``). All fields are
>   explicitly *perceived* — the model is instructed to return ``null`` rather
>   than guess when the image is too low-resolution, partially occluded, or
>   ambiguous, and to include a confidence score.
> - **If the image is not a person** (logo, building, document scan, map, chart,
>   blank): return a brief ``subject_tag`` describing what it is.
>
> Outputs mirror the input layout under
> ``03_processed_outputs/02_gemma_json/<STATE>/<scope>/<jurisdiction>/_contact_images/<name>.json``.

In [ ]:
# Demo 5 — Contact image enrichment. For each image found under a jurisdiction,
# decide whether it is a person; return perceived demographics or a subject tag.
import json as _json

DEMO5_SYSTEM = (
    "You are a careful image triage assistant for a local-government open-data "
    "pipeline. You analyze public contact photos of elected officials and "
    "department heads scraped from public government websites. You always return "
    "strict JSON only. You never claim certainty about demographic attributes — "
    "every attribute is explicitly perceived from the image and you return null "
    "when the image is too low-resolution, partially occluded, ambiguous, or when "
    "you would be guessing rather than observing."
)
DEMO5_USER = (
    "Look at the attached image and decide whether it depicts a single identifiable "
    "person. Return JSON with this shape:\n\n"
    "{\n"
    "  \"is_person\": bool,\n"
    "  \"confidence\": float between 0.0 and 1.0,\n"
    "  \"perceived\": {\n"
    "    \"age_range\": \"one of: child, teen, 20-29, 30-39, 40-49, 50-59, 60-69, 70-plus, or null\",\n"
    "    \"race\": \"perceived racial category or null\",\n"
    "    \"gender\": \"perceived gender presentation or null\",\n"
    "    \"ethnicity\": \"perceived ethnic background or null\",\n"
    "    \"demeanor\": \"one of: formal, smiling, neutral, stern, candid, or null\"\n"
    "  } or null when is_person is false,\n"
    "  \"subject_tag\": \"short descriptor when is_person is false (e.g. 'city logo', "
    "'aerial photo of courthouse', 'organizational chart'); otherwise null\",\n"
    "  \"notes\": \"short caveat — e.g. 'low resolution', 'side profile', "
    "'multiple people visible', 'official portrait crop'\"\n"
    "}\n\n"
    "Rules:\n"
    "- These are PUBLIC officials in a PUBLIC-records transparency context.\n"
    "- Every 'perceived.*' field is a best visual estimate, not a factual claim.\n"
    "- Return null for any 'perceived' subfield where you would be guessing.\n"
    "- If multiple people appear, set is_person=false and use subject_tag='group photo' (and notes).\n"
    "- Return only the JSON object, no prose or markdown."
)


def _mime_for_image(p):
    ext = p.suffix.lower()
    if ext in (".jpg", ".jpeg"): return "image/jpeg"
    if ext == ".png": return "image/png"
    if ext == ".webp": return "image/webp"
    if ext == ".gif": return "image/gif"
    if ext == ".bmp": return "image/bmp"
    return "application/octet-stream"


MAX_IMAGES_PER_JUR = int(os.environ.get("GOVERNANCE_DEMO_MAX_IMAGES_PER_JUR", "12"))

demo5_report = []
for inv in INVENTORIES:
    j = inv.jurisdiction
    if not inv.images:
        continue
    images = inv.images[:MAX_IMAGES_PER_JUR]
    print(f"\n— {j.relative_label}: {len(images)} image(s) (cap={MAX_IMAGES_PER_JUR})")
    for img in images:
        try:
            result = call_google_genai_multimodal(
                api_key=GEMINI_API_KEY,
                model=GENAI_MODEL,
                system_instruction=DEMO5_SYSTEM,
                user_text=DEMO5_USER,
                media=[(img, _mime_for_image(img))],
                temperature=0.0,
                max_output_tokens=1024,
                media_resolution=TOKEN_BUDGET_HIGH,
            )
        except Exception as e:
            print(f"  ! {img.name}: Gemma call failed — {e}")
            continue

        try:
            payload = _json.loads((result.text or "").strip().lstrip("`"))
        except Exception:
            payload = {"_parse_error": True, "_raw": (result.text or "")[:2000]}

        out_path = mirror_output_path(
            input_path=img, raw_root=RAW_ROOT, processed_root=GEMMA_JSON_ROOT,
            suffix=".image_triage.json",
        )
        out_path.write_text(
            _json.dumps(
                {
                    "image": str(img.relative_to(RAW_ROOT)),
                    "jurisdiction_id": j.jurisdiction_id,
                    "model": GENAI_MODEL,
                    "result": payload,
                },
                indent=2, ensure_ascii=False,
            ),
            encoding="utf-8",
        )
        is_person = bool(payload.get("is_person")) if isinstance(payload, dict) else False
        if is_person:
            perceived = payload.get("perceived") or {}
            details = ", ".join(
                f"{k}={v}" for k, v in perceived.items() if v not in (None, "")
            ) or "(no observable attributes)"
            print(f"  · {img.name}: PERSON — {details}")
        else:
            tag = payload.get("subject_tag") if isinstance(payload, dict) else None
            print(f"  · {img.name}: non-person → {tag or '(no tag)'}")
        demo5_report.append(
            {
                "jurisdiction": j.relative_label,
                "jurisdiction_id": j.jurisdiction_id,
                "image": str(img.relative_to(RAW_ROOT)),
                "is_person": is_person,
                "output": str(out_path.relative_to(PROCESSED_ROOT)),
            }
        )

if demo5_report:
    persons = sum(1 for r in demo5_report if r["is_person"])
    print(
        f"\nDemo 5 done: {len(demo5_report)} images classified across "
        f"{len({r['jurisdiction'] for r in demo5_report})} jurisdictions. "
        f"{persons} persons / {len(demo5_report) - persons} non-person subjects."
    )
else:
    print("\nDemo 5: no images in the inventory (check _contact_images/ folders).")

## 11) Optional — merge per-jurisdiction reference data by `jurisdiction_id`

Two parallel reference buckets live under `02_reference_data/`:

- **`meeting_data_by_jurisdiction_id/`** — Orbis exports, registry rows, and any
  other per-jurisdiction reference material (JSON lookups + PDFs sit side-by-side).
  Attached to each analysis under `meeting_data_profile`.
- **`contacts_by_jurisdiction_id/`** — scraped / curated contact rosters keyed
  by `jurisdiction_id` (never `org_id`). Attached under `contacts_profile`.

The cell walks every `*.json` produced by Demo 2 and Demo 3, derives the
`jurisdiction_id` from the mirrored output path, and runs both merges. Any file
matching `*_lookup_by_jurisdiction_id.json` in either bucket is picked up; PDFs
and other artifacts in those folders are left alone as human-readable reference.

In [ ]:
# Walk every *.json under GEMMA_JSON_ROOT and attach the matching meeting-data
# and contacts rows by jurisdiction_id, derived from the mirrored output path.
import json as _json


def _jurisdiction_id_from_output(path):
    """Mirror layout is <STATE>/<scope>/<jur_slug>/...  →  jurisdiction_<state>_<scope>_<tail>."""
    try:
        rel = path.resolve().relative_to(GEMMA_JSON_ROOT.resolve())
    except ValueError:
        return None
    parts = rel.parts
    if len(parts) < 3:
        return None
    state, scope, slug = parts[0].lower(), parts[1].lower(), parts[2].lower()
    scope_prefix = f"{scope}_"
    tail = slug[len(scope_prefix):] if slug.startswith(scope_prefix) else slug
    return f"jurisdiction_{state}_{scope}_{tail}"


meeting_data = load_meeting_data_lookup(PIPE.meeting_data_by_jurisdiction_id)
contacts = load_contacts_lookup(PIPE.contacts_by_jurisdiction_id)

if not meeting_data and not contacts:
    print(
        f"No reference lookups found under {PIPE.meeting_data_by_jurisdiction_id} "
        f"or {PIPE.contacts_by_jurisdiction_id} — skip."
    )
else:
    print(
        f"Loaded {len(meeting_data)} meeting_data and {len(contacts)} contacts "
        "rows keyed by jurisdiction_id."
    )
    merged_meeting = 0
    merged_contacts = 0
    missing_both = 0
    skipped = 0
    for jp in sorted(GEMMA_JSON_ROOT.rglob("*.json")):
        if jp.name.startswith("_") or jp.name.startswith("policy_drift"):
            continue
        try:
            data = _json.loads(jp.read_text(encoding="utf-8"))
        except _json.JSONDecodeError:
            skipped += 1
            continue
        jid = _jurisdiction_id_from_output(jp)
        if not jid:
            continue
        analysis = data
        if isinstance(data, dict) and "json_analysis" in data and isinstance(data["json_analysis"], dict):
            analysis = data["json_analysis"]
        if not isinstance(analysis, dict):
            continue
        had_meeting = jid in meeting_data
        had_contacts = jid in contacts
        if not (had_meeting or had_contacts):
            missing_both += 1
            continue
        if had_meeting:
            merge_meeting_data_by_jurisdiction(analysis, meeting_data, jid)
            merged_meeting += 1
        if had_contacts:
            merge_contacts_by_jurisdiction(analysis, contacts, jid)
            merged_contacts += 1
        jp.write_text(_json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")
        tags = []
        if had_meeting:
            tags.append("meeting_data")
        if had_contacts:
            tags.append("contacts")
        print(f"  enriched: {jp.relative_to(PROCESSED_ROOT)}  jurisdiction_id={jid}  +{','.join(tags)}")
    print(
        f"\nMerge done: meeting_data={merged_meeting}, contacts={merged_contacts}, "
        f"no-match jurisdictions={missing_both}, unparseable files={skipped}."
    )

## 12) Troubleshooting

- **`gemma-4-…` model id not found:** open AI Studio, copy a Gemma 4 id your
  account can use, and set `GOVERNANCE_GENAI_MODEL` to that string.
- **`ThinkingConfig` / `MediaResolution` AttributeError:** upgrade
  `google-genai` (`%pip install -q -U "google-genai>=1.0"`). Older builds expose
  the older API surface only; the helper falls back silently when the enum is
  missing but you'll see the model default budget instead of HIGH/LOW.
- **`ffmpeg: command not found` in Demo 4:** Colab ships ffmpeg by default; locally
  install with `apt-get install -y ffmpeg` (Linux) or `brew install ffmpeg` (macOS).
- **`PyMuPDF` import error:** re-run the install cell — `pymupdf` must be present
  for Demo 1 and Demo 2.
- **Payload too big for one request:** reduce `GOVERNANCE_DEMO_MAX_PAGES_PER_PDF`
  or use the Files API (not wired up here) for very long audio.
- **Quota / cost spikes:** lower the demo caps (`MAX_PDFS_PER_JUR`, `MAX_AUDIO_CHUNKS`)
  while you iterate.